In [8]:
import numpy as np
import ml_dtypes

In [32]:
def generate_cbfloat16_lut_mldtypes(num_points=512):
    out = ""
    # 1. Generate angles array (k = 0 to num_points - 1)
    k = np.arange(num_points)
    angles = 2.0 * np.pi * k / num_points
    
    # 2. Calculate real and imaginary parts in standard float32
    real_f32 = np.cos(angles).astype(np.float32)
    imag_f32 = np.sin(angles).astype(np.float32)
    
    # 3. Convert directly to bfloat16 using ml_dtypes, 
    # then cast back to float32 to extract the exact truncated C++ literal
    real_bf16 = real_f32.astype(ml_dtypes.bfloat16).astype(np.float32)
    imag_bf16 = imag_f32.astype(ml_dtypes.bfloat16).astype(np.float32)
    
    # 4. Write to C++ header
    out = out + f"// Auto-generated {num_points}-point e^(ix) LUT for cbfloat16\n"
    out = out + f"// Generated using numpy and ml_dtypes\n"
    out = out + f"#pragma once\n"
    out = out + f"#include <aie_api/aie_types.hpp>\n\n"
    out = out + f"#DEFINE LUT_SIZE {num_points} \n\n"
    
    out = out + f"alignas(32) const cbfloat16 freq_shift_lut[LUT_SIZE] = {{\n"
    
    for i in range(num_points):
        separator = "," if i < num_points - 1 else ""
        # The AIE compiler expects standard float literals (e.g., 1.0f) which it will cast down
        out = out + f"    {{ (bfloat16){real_bf16[i]}f, (bfloat16){imag_bf16[i]}f }}{separator} // k={i}\n"
        
    out = out + "};\n"

    return out


In [33]:
print(generate_cbfloat16_lut_mldtypes(512))

// Auto-generated 512-point e^(ix) LUT for cbfloat16
// Generated using numpy and ml_dtypes
#pragma once
#include <aie_api/aie_types.hpp>

#DEFINE LUT_SIZE 512 

alignas(32) const cbfloat16 freq_shift_lut[LUT_SIZE] = {
    { (bfloat16)1.0f, (bfloat16)0.0f }, // k=0
    { (bfloat16)1.0f, (bfloat16)0.01226806640625f }, // k=1
    { (bfloat16)1.0f, (bfloat16)0.0245361328125f }, // k=2
    { (bfloat16)1.0f, (bfloat16)0.036865234375f }, // k=3
    { (bfloat16)1.0f, (bfloat16)0.049072265625f }, // k=4
    { (bfloat16)1.0f, (bfloat16)0.061279296875f }, // k=5
    { (bfloat16)0.99609375f, (bfloat16)0.07373046875f }, // k=6
    { (bfloat16)0.99609375f, (bfloat16)0.0859375f }, // k=7
    { (bfloat16)0.99609375f, (bfloat16)0.09814453125f }, // k=8
    { (bfloat16)0.9921875f, (bfloat16)0.1103515625f }, // k=9
    { (bfloat16)0.9921875f, (bfloat16)0.12255859375f }, // k=10
    { (bfloat16)0.9921875f, (bfloat16)0.134765625f }, // k=11
    { (bfloat16)0.98828125f, (bfloat16)0.146484375f }, // k=12
  